# Lab Session 2 — Knowledge Base Construction, Alignment & Expansion
**Domain:** Michael Jackson et Wikidata  

1. **Step 1** : Construction de la KB privée initiale (RDF)
2. **Step 2** : Entity Linking avec Wikidata (owl:sameAs)
3. **Step 3** : Alignement des prédicats via SPARQL
4. **Step 4** : Expansion de la KB via SPARQL + nettoyage


## Installation


In [ ]:
!pip install rdflib SPARQLWrapper pandas requests

---
## Step 1 — Construction de la KB privée initiale

On charge les fichiers produits au Lab 1 (`triplets_initiaux.csv` et `entities_mJackson.csv`) et on les convertit en graphe RDF avec **rdflib**.  

- Chaque entité devient un `URIRef` (pas une simple chaîne)
- Les prédicats sont normalisés en camelCase
- On utilise deux namespaces séparés : `mj:` pour les entités, `prop:` pour les prédicats
- On évite les doublons via `g.add()` qui est idempotent


In [ ]:
import pandas as pd
import re
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import OWL, RDFS, RDF

# Namespaces
MJ   = Namespace('http://mjackson.org/resource/')
PROP = Namespace('http://mjackson.org/property/')

g = Graph()
g.bind('mj',   MJ)
g.bind('prop', PROP)
g.bind('owl',  OWL)
g.bind('rdfs', RDFS)

def clean_uri(text):
    """Transforme un texte en fragment d'URI valide."""
    text = str(text).strip()
    text = text.replace(' ', '_')
    text = re.sub(r'[^\w\-_]', '', text)
    return text

def to_camel_case(text):
    """Normalise un prédicat en camelCase (ex: is_related_to → isRelatedTo)."""
    parts = str(text).strip().split('_')
    return parts[0].lower() + ''.join(p.capitalize() for p in parts[1:])

# Chargement des fichiers du Lab 1
triplets_df  = pd.read_csv('triplets_initiaux.csv')
entities_df  = pd.read_csv('entities_mJackson.csv')

print(f'Triplets chargés  : {len(triplets_df)}')
print(f'Entités chargées  : {len(entities_df)}')


In [3]:
# Construction du graphe RDF
added = 0
seen  = set()

for _, row in triplets_df.iterrows():
    s = clean_uri(row['subject'])
    p = to_camel_case(row['predicate'])
    o = clean_uri(row['object'])

    # Filtre : entités trop courtes ou trop longues
    if len(s) < 2 or len(o) < 2 or len(s) > 80 or len(o) > 80:
        continue

    triple = (s, p, o)
    if triple in seen:
        continue
    seen.add(triple)

    g.add((
        URIRef(MJ   + s),
        URIRef(PROP + p),
        URIRef(MJ   + o),
    ))
    added += 1

print(f'Triplets ajoutés dans la KB privée : {added}')
print(f'Taille du graphe RDF               : {len(g)} triplets')


Triplets ajoutés dans la KB privée : 5162
Taille du graphe RDF               : 5162 triplets


In [4]:
ents  = set()
preds = set()
for s, p, o in g:
    ents.add(s); ents.add(o); preds.add(p)

print('--- Statistiques KB privée initiale ---')
print(f'  Triplets  : {len(g):>6}  (min requis : 100)')
print(f'  Entités   : {len(ents):>6}  (min requis : 50)')
print(f'  Prédicats : {len(preds):>6}')
print(f'  Critères  : {"OK" if len(g) >= 100 and len(ents) >= 50 else "KO"}')


--- Statistiques KB privée initiale ---
  Triplets  :   5162  (min requis : 100)
  Entités   :   1048  (min requis : 50)
  Prédicats :     39
  Critères  : OK



## Step 2 — Entity Linking avec Wikidata

Pour chaque entité de notre KB privée, on interroge l'API MediaWiki de Wikidata pour trouver le QID correspondant 

Deux cas possibles :
- Entité trouvée: on ajoute mj:X owl:sameAs wd:QXXX
- Entité non trouvée → on la définit localement avec rdf:type et rdfs:label

Calcul du score de confiance :
- 1.0 si le label Wikidata correspond exactement au texte
- 0.8 si correspondance approchée
- 0.0 si aucun résultat


In [5]:
import requests
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 MJacksonKB/1.0 (student project; contact@example.com)"
}

def search_wikidata(entity_name):
    for lang in ['en', 'fr']:
        try:
            resp = requests.get(
                'https://www.wikidata.org/w/api.php',
                params={
                    'action':   'wbsearchentities',
                    'search':   entity_name,
                    'language': lang,
                    'limit':    1,
                    'format':   'json'
                },
                headers=HEADERS,   
                timeout=15
            )
            results = resp.json().get('search', [])
            if results:
                hit   = results[0]
                label = hit.get('label', '')
                if label == entity_name:
                    score = 1.0
                elif label.lower() == entity_name.lower():
                    score = 0.8
                else:
                    score = 0.0
                if score > 0:
                    return hit['id'], score
        except Exception as e:
            print(f'  API error for "{entity_name}": {e}')
        time.sleep(0.1)
    return None, 0.0

In [6]:

unique_entities = list(set(
    triplets_df['subject'].tolist() + triplets_df['object'].tolist()
))

unique_entities = [
    e for e in unique_entities
    if 2 < len(str(e)) <= 80 and not str(e).isnumeric()
]

print(f'{len(unique_entities)} entités uniques à aligner')


1052 entités uniques à aligner


In [7]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import OWL, RDFS

MJ   = Namespace("http://mjackson.org/resource/")
PROP = Namespace("http://mjackson.org/property/")
WD   = Namespace("http://www.wikidata.org/entity/")
WDT  = Namespace("http://www.wikidata.org/prop/direct/")

import re
def clean_uri(text):
    text = str(text).strip()
    text = text.replace(' ', '_')
    text = re.sub(r'[^\w\-_]', '', text)
    return text

print("Namespaces redéfinis ✓")

Namespaces redéfinis ✓


In [8]:
mapping       = []
aligned_qids  = {}   

print(f'Alignement de {len(unique_entities)} entités...')
print('(cela peut prendre quelques minutes)\n')

for i, entity in enumerate(unique_entities):
    qid, score = search_wikidata(entity)
    private_uri = URIRef(MJ + clean_uri(entity))

    if qid and score >= 0.8:
        wikidata_uri = URIRef(WD + qid)
        g.add((private_uri, OWL.sameAs, wikidata_uri))
        aligned_qids[entity] = qid
        status = f'✓ {qid}  (score={score})'
    else:
        
        label_row = entities_df[entities_df['entity'] == entity]
        if not label_row.empty:
            ltype = label_row.iloc[0]['label']
        else:
            ltype = 'ORG'
        type_map = {
            'PERSON': 'http://schema.org/Person',
            'ORG':    'http://schema.org/Organization',
            'GPE':    'http://schema.org/Place',
            'MISC':   'http://schema.org/Thing',
        }
        g.add((private_uri, RDF.type,    URIRef(type_map.get(ltype, 'http://schema.org/Thing'))))
        g.add((private_uri, RDFS.label,  Literal(entity)))
        status = '✗ non trouvé — défini localement'

    mapping.append({
        'private_entity': str(private_uri),
        'wikidata_qid':   qid or '',
        'wikidata_uri':   f'http://www.wikidata.org/entity/{qid}' if qid else '',
        'confidence':     score,
    })

    if (i + 1) % 50 == 0:
        n_ok = sum(1 for m in mapping if m['confidence'] >= 0.8)
        print(f'  [{i+1}/{len(unique_entities)}] alignées : {n_ok}')

alignment_df = pd.DataFrame(mapping)
alignment_df.to_csv('entity_alignment.csv', index=False)

n_aligned = sum(1 for m in mapping if m['confidence'] >= 0.8)
print(f'\nEntités alignées sur Wikidata : {n_aligned}/{len(unique_entities)}')
print('Fichier entity_alignment.csv sauvegardé.')


Alignement de 1052 entités...
(cela peut prendre quelques minutes)

  [50/1052] alignées : 19
  [100/1052] alignées : 45
  [150/1052] alignées : 73
  [200/1052] alignées : 96
  [250/1052] alignées : 120
  [300/1052] alignées : 146
  [350/1052] alignées : 169
  [400/1052] alignées : 200
  [450/1052] alignées : 230
  [500/1052] alignées : 252
  [550/1052] alignées : 280
  [600/1052] alignées : 309
  [650/1052] alignées : 337
  [700/1052] alignées : 363
  [750/1052] alignées : 392
  [800/1052] alignées : 419
  [850/1052] alignées : 439
  [900/1052] alignées : 460
  [950/1052] alignées : 486
  [1000/1052] alignées : 511
  [1050/1052] alignées : 539

Entités alignées sur Wikidata : 541/1052
Fichier entity_alignment.csv sauvegardé.


In [9]:
print('--- Table dalignement (extrait) ---')
print(alignment_df[alignment_df['confidence'] >= 0.8]
      [['private_entity','wikidata_qid','confidence']]
      .head(20).to_string())


--- Table dalignement (extrait) ---
                                           private_entity wikidata_qid  confidence
1                      http://mjackson.org/resource/Janet    Q13553631         1.0
2   http://mjackson.org/resource/The_Way_You_Make_Me_Feel    Q93963144         1.0
6                    http://mjackson.org/resource/Jackson     Q2732758         1.0
7                   http://mjackson.org/resource/AEG_Live      Q975408         1.0
10               http://mjackson.org/resource/John_Lennon        Q1203         1.0
12                  http://mjackson.org/resource/Carrière    Q21482747         1.0
17                       http://mjackson.org/resource/CBS       Q43380         1.0
19               http://mjackson.org/resource/Butterflies      Q832368         1.0
25               http://mjackson.org/resource/ITV_Granada     Q2476957         1.0
28                     http://mjackson.org/resource/Xanax    Q47521014         1.0
29     http://mjackson.org/resource/Madison_Square_


## Step 3 — Alignement des prédicats via SPARQL
On aligne nos prédicats privés avec des propriétés Wikidata 

- Par les paires d'entités alignées : Pour une paire (s, o) bien alignée, on interroge Wikidata pour voir quelles propriétés relient wiki(s) à wiki(o) directement.

- Par recherche textuelle : On cherche les propriétés Wikidata dont le label contient des mots-clés sémantiquement proches de nos prédicats (related, award, genre...).
- on compare les candidats avec nos prédicats et on choisit l'alignement le plus pertinent

- `owl:equivalentProperty` si les deux propriétés ont exactement le même sens
- `rdfs:subPropertyOf` si notre propriété est plus spécifique


In [10]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper('https://query.wikidata.org/sparql')
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader(
    'User-Agent', 'MJacksonKB/1.0 (student project)'
)

def sparql_query(query):
    """Exécute une requête SPARQL sur Wikidata, retourne les bindings."""
    try:
        sparql.setQuery(query)
        return sparql.query().convert()['results']['bindings']
    except Exception as e:
        print(f' SPARQL error: {e}')
        return []


In [11]:
# Méthode 1 : propriétés entre paires d'entités alignées (Bidirectionnel) 

KEY_PAIRS = [
    ('Michael Jackson', 'Q2831', 'Epic Records',  'Q193023'),
    ('Michael Jackson', 'Q2831', 'Diana Ross',    'Q161819'),
    ('Michael Jackson', 'Q2831', 'Stevie Wonder', 'Q210379'),
    ('Michael Jackson', 'Q2831', 'Jackson Five',  'Q217296'),
    ('Michael Jackson', 'Q2831', 'Los Angeles',   'Q65'),
]

print('Propriétés Wikidata pour les paires clés :')
prop_candidates = {}

for s_name, qid_s, o_name, qid_o in KEY_PAIRS:
    query = f"""
    SELECT ?prop ?propLabel ?dir WHERE {{
      {{
        # Sens normal (Sujet → Objet)
        wd:{qid_s} ?prop wd:{qid_o} .
        BIND("→" AS ?dir)
      }}
      UNION
      {{
        # Sens inverse (Objet → Sujet)
        wd:{qid_o} ?prop wd:{qid_s} .
        BIND("←" AS ?dir)
      }}
      ?propEntity wikibase:directClaim ?prop .
      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en" .
        ?propEntity rdfs:label ?propLabel .
      }}
    }} LIMIT 10
    """
    
    results = sparql_query(query)
    
    # On récupère l'URI le label brut et la flèche
    props = [(
        r['prop']['value'], 
        r.get('propLabel', {}).get('value', ''),
        r.get('dir', {}).get('value', '')
    ) for r in results]
    
    print(f'  {s_name} ↔ {o_name} : {[f"{p[2]} {p[1]}" for p in props]}')
    
    for uri, label, dir_arrow in props:
        if label:
            prop_candidates[label] = uri
            
    time.sleep(1)

Propriétés Wikidata pour les paires clés :
  Michael Jackson ↔ Epic Records : []
  Michael Jackson ↔ Diana Ross : []
  Michael Jackson ↔ Stevie Wonder : []
  Michael Jackson ↔ Jackson Five : []
  Michael Jackson ↔ Los Angeles : ['→ place of death']


In [12]:
# Méthode 2 : recherche textuelle sur les mots-clés de nos prédicats 

KEYWORDS = ['related', 'influenced', 'award', 'record label',
            'genre', 'member of', 'performer', 'notable work']

print('Recherche textuelle de propriétés Wikidata :')

for kw in KEYWORDS:
    query = f"""
    SELECT ?prop ?propLabel WHERE {{
      ?prop a wikibase:Property .
      ?prop rdfs:label ?propLabel .
      FILTER(LANG(?propLabel) = "en")
      FILTER(CONTAINS(LCASE(?propLabel), "{kw}"))
    }} LIMIT 5
    """
    results = sparql_query(query)
    for r in results:
        uri   = r['prop']['value']
        label = r.get('propLabel', {}).get('value', '')
        prop_candidates[label] = uri
        print(f'  [{kw}] {label} → {uri}')
    time.sleep(0.5)


Recherche textuelle de propriétés Wikidata :
  [related] related image → http://www.wikidata.org/entity/P6802
  [related] related Wikidata property → http://www.wikidata.org/entity/P1659
  [related] organized response related to outbreak → http://www.wikidata.org/entity/P8045
  [related] list related to category → http://www.wikidata.org/entity/P1753
  [related] category related to list → http://www.wikidata.org/entity/P1754
  [influenced] influenced by → http://www.wikidata.org/entity/P737
  [award] points awarded → http://www.wikidata.org/entity/P3260
  [award] award received → http://www.wikidata.org/entity/P166
  [award] César Award film ID → http://www.wikidata.org/entity/P5318
  [award] César Award person ID → http://www.wikidata.org/entity/P5319
  [award] awarded for period → http://www.wikidata.org/entity/P4566
  [record label] record label → http://www.wikidata.org/entity/P264
  [genre] genre → http://www.wikidata.org/entity/P136
  [genre] Library of Congress Genre/Form Terms 

In [13]:
# ── Construction de l'ontologie d'alignement ──
# Validation manuelle

ontology = Graph()
ontology.bind('prop', PROP)
ontology.bind('owl',  OWL)
ontology.bind('rdfs', RDFS)
ontology.bind('wdt',  WDT)

# Propriétés Wikidata validées manuellement
# Format : (notre_predicat, wikidata_uri, type_alignement)
# type : equivalent ou sub
VALIDATED_ALIGNMENTS = [
    # is_related_to est très générique → les props Wikidata en sont des sous-propriétés
    ('isRelatedTo',    'http://www.wikidata.org/prop/direct/P737',  'sub'),   #influenced by
    ('isRelatedTo',    'http://www.wikidata.org/prop/direct/P463',  'sub'),   # member of
    ('isRelatedTo',    'http://www.wikidata.org/prop/direct/P264',  'sub'),   #record label
    # Prédicats sémantiques extraits par le dependency parsing
    ('signer',         'http://www.wikidata.org/prop/direct/P264',  'equivalent'),  #record label
    ('recevoir',       'http://www.wikidata.org/prop/direct/P166',  'equivalent'),  #award received
    ('sortir',         'http://www.wikidata.org/prop/direct/P1344', 'sub'),   #participant in
    ('rejoindre',      'http://www.wikidata.org/prop/direct/P463',  'equivalent'),  # member of
    ('enregistrer',    'http://www.wikidata.org/prop/direct/P358',  'equivalent'),  #discography
]

for our_pred, wd_uri, align_type in VALIDATED_ALIGNMENTS:
    our_uri = URIRef(PROP + our_pred)
    wd_prop = URIRef(wd_uri)
    if align_type == 'equivalent':
        ontology.add((our_uri, OWL.equivalentProperty, wd_prop))
    else:
        ontology.add((wd_prop, RDFS.subPropertyOf, our_uri))
    ontology.add((our_uri, RDFS.label, Literal(our_pred, lang='en')))

ontology.serialize('ontology_alignment.ttl', format='turtle')
print(f'Ontologie sauvegardée : {len(ontology)} triplets')
print('Fichier : ontology_alignment.ttl')


Ontologie sauvegardée : 14 triplets
Fichier : ontology_alignment.ttl


---
## Step 4 — Expansion de la KB via SPARQL

**Objectif final :** 50 000 – 200 000 triplets, 5 000 – 30 000 entités, 50 – 200 relations.

**Stratégie d'expansion ancrée :**  
On ne part que des entités **confidently aligned** (score ≥ 0.8). Partir d'entités mal alignées introduirait des triplets sans rapport avec Michael Jackson.

**Trois niveaux d'expansion :**
1. **1-hop** : tous les triplets directs de chaque entité alignée sur Wikidata
2. **Predicate-controlled** : expansion ciblée sur les propriétés les plus importantes
3. **2-hop** : triplets des entités découvertes au 1-hop (ex: les albums, les labels)


In [14]:
# Entités bien alignées (score >= 0.8) → base de l'expansion
high_conf = alignment_df[alignment_df['confidence'] >= 0.8]
print(f'Entités bien alignées : {len(high_conf)}')
print('Exemples :')
print(high_conf[['private_entity','wikidata_qid','confidence']].head(15).to_string())


Entités bien alignées : 541
Exemples :
                                           private_entity wikidata_qid  confidence
1                      http://mjackson.org/resource/Janet    Q13553631         1.0
2   http://mjackson.org/resource/The_Way_You_Make_Me_Feel    Q93963144         1.0
6                    http://mjackson.org/resource/Jackson     Q2732758         1.0
7                   http://mjackson.org/resource/AEG_Live      Q975408         1.0
10               http://mjackson.org/resource/John_Lennon        Q1203         1.0
12                  http://mjackson.org/resource/Carrière    Q21482747         1.0
17                       http://mjackson.org/resource/CBS       Q43380         1.0
19               http://mjackson.org/resource/Butterflies      Q832368         1.0
25               http://mjackson.org/resource/ITV_Granada     Q2476957         1.0
28                     http://mjackson.org/resource/Xanax    Q47521014         1.0
29     http://mjackson.org/resource/Madison_Squa

In [15]:
def expand_entity_1hop(qid, limit=1000):
    """
    Récupère tous les triplets directs d'une entité Wikidata (1-hop).
    On filtre pour ne garder que les objets URI (pas les littéraux).
    """
    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{qid} ?p ?o .
      FILTER(isIRI(?o))
    }} LIMIT {limit}
    """
    results = sparql_query(query)
    return [
        (r['p']['value'], r['o']['value'])
        for r in results
    ]

# Expansion 1-hop sur toutes les entités bien alignées
print('Expansion 1-hop...')
before = len(g)

discovered_entities = set()  #entités découvertes pendant l'expansion

for i, (_, row) in enumerate(high_conf.iterrows()):
    qid     = row['wikidata_qid']
    wd_uri  = URIRef(WD + qid)

    triples = expand_entity_1hop(qid, limit=1000)
    for p_uri, o_uri in triples:
        o_ref = URIRef(o_uri)
        g.add((wd_uri, URIRef(p_uri), o_ref))
        #on mémorise les nouvelles entités pour le 2-hop
        if o_uri.startswith('http://www.wikidata.org/entity/Q'):
            discovered_entities.add(o_uri.split('/')[-1])

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/{len(high_conf)}] KB size : {len(g)} triplets')
    time.sleep(0.5)

print(f'\n1-hop terminé : {len(g) - before} triplets ajoutés')
print(f'Nouvelles entités découvertes : {len(discovered_entities)}')


Expansion 1-hop...
  [10/541] KB size : 7845 triplets
  [20/541] KB size : 9422 triplets
  [30/541] KB size : 10397 triplets
  [40/541] KB size : 12173 triplets
  [50/541] KB size : 12842 triplets
  [60/541] KB size : 14250 triplets
  [70/541] KB size : 14870 triplets
  [80/541] KB size : 16756 triplets
  [90/541] KB size : 18032 triplets
  [100/541] KB size : 19863 triplets
  [110/541] KB size : 20763 triplets
  [120/541] KB size : 22142 triplets
  [130/541] KB size : 24319 triplets
  [140/541] KB size : 25900 triplets
  [150/541] KB size : 28294 triplets
  [160/541] KB size : 28725 triplets
  [170/541] KB size : 29285 triplets
  [180/541] KB size : 31441 triplets
  [190/541] KB size : 32512 triplets
  [200/541] KB size : 32863 triplets
  [210/541] KB size : 34295 triplets
  [220/541] KB size : 34963 triplets
  [230/541] KB size : 36623 triplets
  [240/541] KB size : 37739 triplets
  [250/541] KB size : 38630 triplets
  [260/541] KB size : 40605 triplets
  [270/541] KB size : 41805 tr

In [16]:
# Expansion predicate-controlled 
# On cible les propriétés les plus importantes pour notre domaine

TARGET_PROPS = {
    'P166':  'award received',
    'P264':  'record label',
    'P136':  'genre',
    'P463':  'member of',
    'P737':  'influenced by',
    'P57':   'director',
    'P175':  'performer',
    'P495':  'country of origin',
    'P407':  'language of work',
    'P577':  'publication date',
}

print('Expansion predicate-controlled...')
before = len(g)

for pid, plabel in TARGET_PROPS.items():
    query = f"""
    SELECT ?s ?o WHERE {{
      ?s wdt:{pid} ?o .
      FILTER(isIRI(?o))
    }} LIMIT 5000
    """
    results = sparql_query(query)
    for r in results:
        s = URIRef(r['s']['value'])
        o = URIRef(r['o']['value'])
        g.add((s, WDT[pid], o))
    print(f'  P{pid} ({plabel}) : {len(results)} triplets ajoutés')
    time.sleep(1)

print(f'\nPredicate-controlled terminé : {len(g) - before} triplets ajoutés')


Expansion predicate-controlled...
  PP166 (award received) : 5000 triplets ajoutés
  PP264 (record label) : 5000 triplets ajoutés
  PP136 (genre) : 5000 triplets ajoutés
  PP463 (member of) : 5000 triplets ajoutés
  PP737 (influenced by) : 5000 triplets ajoutés
  PP57 (director) : 5000 triplets ajoutés
  PP175 (performer) : 5000 triplets ajoutés
  PP495 (country of origin) : 5000 triplets ajoutés
  PP407 (language of work) : 5000 triplets ajoutés
  PP577 (publication date) : 1760 triplets ajoutés

Predicate-controlled terminé : 46666 triplets ajoutés


In [17]:
# expansion 2-hop
#on récupère les triplets des entités découvertes au 1-hop
# On limite à 500 entités pour rester dans les contraintes mémoire

print('Expansion 2-hop...')
before = len(g)

# On priorise les QIDs les plus souvent rencontrés au 1-hop
sample_2hop = list(discovered_entities)[:500]

for i, qid in enumerate(sample_2hop):
    triples = expand_entity_1hop(qid, limit=200)
    wd_uri  = URIRef(WD + qid)
    for p_uri, o_uri in triples:
        g.add((wd_uri, URIRef(p_uri), URIRef(o_uri)))

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(sample_2hop)}] KB size : {len(g)} triplets')
    time.sleep(0.3)

print(f'\n2-hop terminé : {len(g) - before} triplets ajoutés')
print(f'Taille totale du graphe : {len(g)} triplets')


Expansion 2-hop...
  [50/500] KB size : 133465 triplets
  [100/500] KB size : 136466 triplets
  ⚠ SPARQL error: HTTP Error 502: Bad Gateway
  [150/500] KB size : 139001 triplets
  [200/500] KB size : 142289 triplets
  [250/500] KB size : 145105 triplets
  [300/500] KB size : 148686 triplets
  [350/500] KB size : 151539 triplets
  [400/500] KB size : 154311 triplets
  [450/500] KB size : 157045 triplets
  [500/500] KB size : 160268 triplets

2-hop terminé : 30344 triplets ajoutés
Taille totale du graphe : 160268 triplets


### Nettoyage avant export

- Doublons
- on supprime les propriétés qui génèrent beaucoup de littéraux (dates, descriptions, labels) 
- on vérifie qu'il n'y a pas de nœuds sans connexion


In [18]:
from rdflib import URIRef
from collections import Counter

print("Nettoyage du graphe...")

rel_counter = Counter()
for s, p, o in g:
    if isinstance(o, Literal): continue
    if "statement/" in str(s) or "statement/" in str(o): continue
    if "/prop/direct/" not in str(p): continue
    rel_counter[str(p)] += 1

KEPT_RELATIONS = {uri for uri, count in rel_counter.items() if count >= 50}
print(f"Relations fréquentes (≥50 utilisations) : {len(KEPT_RELATIONS)}")

g_clean = Graph()
g_clean.bind('mj',  MJ)
g_clean.bind('wd',  WD)
g_clean.bind('wdt', WDT)

for s, p, o in g:
    #filtre littéraux
    if isinstance(o, Literal):
        continue
    #filtre statements Wikidata parasites
    if "statement/" in str(s) or "statement/" in str(o):
        continue
    #filtre prédicats non-directs
    if "/prop/direct/" not in str(p):
        continue
    #filtre relations fréquentes
    if str(p) not in KEPT_RELATIONS:
        continue
    #filtre QIDs et entités privées uniquement
    s_str = str(s)
    o_str = str(o)
    valid_s = "wikidata.org/entity/Q" in s_str or "mjackson.org" in s_str
    valid_o = "wikidata.org/entity/Q" in o_str or "mjackson.org" in o_str
    if not (valid_s and valid_o):
        continue
    g_clean.add((s, p, o))


ents_clean = set()
rels_clean = set()
for s, p, o in g_clean:
    ents_clean.add(str(s))
    ents_clean.add(str(o))
    rels_clean.add(str(p))

ok_t = 50_000 <= len(g_clean)    <= 200_000
ok_e = 5_000  <= len(ents_clean) <= 30_000
ok_r = 50     <= len(rels_clean) <= 200

print("=" * 55)
print("STATISTIQUES APRÈS NETTOYAGE")
print("=" * 55)
print(f"  Triplets  : {len(g_clean):>8}   (requis : 50k–200k)")
print(f"  Entités   : {len(ents_clean):>8}   (requis : 5k–30k)")
print(f"  Relations : {len(rels_clean):>8}   (requis : 50–200)")
print("=" * 55)
print(f"  {'OK' if ok_t else 'KO'} Triplets")
print(f"  {'OK' if ok_e else 'KO'} Entités")
print(f"  {'OK' if ok_r else 'KO'} Relations")

# ── 4. Export ──
g_clean.serialize("expanded_kb.nt", format="nt")
print("\nexpanded_kb.nt sauvegardé (graphe nettoyé)")

Nettoyage du graphe...
Relations fréquentes (≥50 utilisations) : 108
STATISTIQUES APRÈS NETTOYAGE
  Triplets  :    66719   (requis : 50k–200k)
  Entités   :    54768   (requis : 5k–30k)
  Relations :       98   (requis : 50–200)
  OK Triplets
  KO Entités
  OK Relations


C:\Users\laura\Anaconda4\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(



expanded_kb.nt sauvegardé (graphe nettoyé)


### Export des fichiers finaux

- `expanded_kb.nt` : le graphe complet au format N-Triples
- `ontology_alignment.ttl` : l'ontologie et les alignements de prédicats
- `entity_alignment.csv` : la table d'alignement entités privées ↔ Wikidata


In [21]:
g_clean.serialize("expanded_kb.nt", format="nt")
print("Fichier filtré sauvegardé avec succès !")
print('ontology_alignment.ttl sauvegardé')

print('entity_alignment.csv sauvegardé')

print('\n=== Fichiers produits pour le Lab 3 ===')
print('  -> expanded_kb.nt          (graphe RDF étendu)')
print('  -> ontology_alignment.ttl  (alignement des prédicats)')
print('  -> entity_alignment.csv    (alignement des entités)')


C:\Users\laura\Anaconda4\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Fichier filtré sauvegardé avec succès !
ontology_alignment.ttl sauvegardé
entity_alignment.csv sauvegardé

=== Fichiers produits pour le Lab 3 ===
  -> expanded_kb.nt          (graphe RDF étendu)
  -> ontology_alignment.ttl  (alignement des prédicats)
  -> entity_alignment.csv    (alignement des entités)


In [22]:
import pandas as pd
from rdflib import Graph, URIRef, Literal
from collections import Counter

print("=" * 60)
print("RAPPORT DE STATISTIQUES — KB Michael Jackson × Wikidata")
print("=" * 60)

# ── 1. KB PRIVÉE INITIALE ──
print("\n--- 1. KB PRIVÉE INITIALE (Lab 1) ---")
df_t = pd.read_csv("triplets_initiaux.csv")
df_e = pd.read_csv("entities_mJackson.csv")

n_triplets_init  = len(df_t)
n_entities_init  = len(df_e["entity"].unique())
n_predicats_init = len(df_t["predicate"].unique())
n_sources        = len(df_e["url"].unique())

print(f"  Triplets extraits      : {n_triplets_init}")
print(f"  Entités uniques        : {n_entities_init}")
print(f"  Prédicats uniques      : {n_predicats_init}")
print(f"  Sources crawlées       : {n_sources}")
print(f"  Répartition entités    :")
for label, count in df_e["label"].value_counts().items():
    print(f"    {label:>8} : {count}")
print(f"  Top 5 prédicats        :")
for pred, count in df_t["predicate"].value_counts().head(5).items():
    print(f"    {pred:>20} : {count}")

# ── 2. ALIGNEMENT DES ENTITÉS ──
print("\n--- 2. ALIGNEMENT DES ENTITÉS (Step 2) ---")
df_align = pd.read_csv("entity_alignment.csv")

n_total    = len(df_align)
n_aligned  = len(df_align[df_align["confidence"] >= 0.8])
n_exact    = len(df_align[df_align["confidence"] == 1.0])
n_approx   = len(df_align[(df_align["confidence"] >= 0.8) & (df_align["confidence"] < 1.0)])
n_notfound = len(df_align[df_align["confidence"] == 0.0])
pct        = round(n_aligned / n_total * 100, 1)

print(f"  Entités soumises à l'alignement : {n_total}")
print(f"  Alignées (score ≥ 0.8)          : {n_aligned} ({pct}%)")
print(f"    dont score exact (1.0)         : {n_exact}")
print(f"    dont score approché (0.8)      : {n_approx}")
print(f"  Non trouvées (score = 0.0)       : {n_notfound}")
print(f"  Taux d'alignement                : {pct}%")

# ── 3. ALIGNEMENT DES PRÉDICATS ──
print("\n--- 3. ALIGNEMENT DES PRÉDICATS (Step 3) ---")
onto = Graph()
onto.parse("ontology_alignment.ttl", format="turtle")

from rdflib.namespace import OWL, RDFS
n_equiv = len(list(onto.triples((None, OWL.equivalentProperty, None))))
n_sub   = len(list(onto.triples((None, RDFS.subPropertyOf, None))))

print(f"  Triplets dans l'ontologie        : {len(onto)}")
print(f"  owl:equivalentProperty           : {n_equiv}")
print(f"  rdfs:subPropertyOf               : {n_sub}")
print(f"  Prédicats privés alignés         :")
for s, p, o in onto.triples((None, OWL.equivalentProperty, None)):
    print(f"    {str(s).split('/')[-1]:>25} → {str(o).split('/')[-1]}")

# ── 4. KB ÉTENDUE ──
print("\n--- 4. KB ÉTENDUE FINALE (Step 4) ---")
g_final = Graph()
g_final.parse("expanded_kb.nt", format="nt")

ents_final = set()
rels_final = set()
rel_counter_final = Counter()

for s, p, o in g_final:
    if not isinstance(o, Literal):
        ents_final.add(str(s))
        ents_final.add(str(o))
        rels_final.add(str(p))
        rel_counter_final[str(p)] += 1

n_t = len(g_final)
n_e = len(ents_final)
n_r = len(rels_final)

print(f"  Triplets totaux                  : {n_t}")
print(f"  Entités uniques                  : {n_e}")
print(f"  Relations uniques                : {n_r}")
print(f"  Densité moyenne (triplets/entité): {round(n_t / n_e, 2)}")

print(f"\n  Critères du sujet :")
print(f"    {'OK' if 50_000  <= n_t <= 200_000 else 'KO'} Triplets  : {n_t:>8}  (requis : 50k–200k)")
print(f"    {'OK' if 5_000   <= n_e <= 30_000  else 'KO'} Entités   : {n_e:>8}  (requis : 5k–30k)")
print(f"    {'OK' if 50      <= n_r <= 200     else 'KO'} Relations : {n_r:>8}  (requis : 50–200)")

print(f"\n  Top 10 relations les plus fréquentes :")
for uri, count in rel_counter_final.most_common(10):
    pid = uri.split("/")[-1]
    print(f"    {pid:>10} : {count:>6} triplets")

# ── 5. COMPARAISON AVANT / APRÈS EXPANSION ──
print("\n--- 5. COMPARAISON AVANT / APRÈS EXPANSION ---")
print(f"  {'Métrique':<25} {'Avant':>10} {'Après':>10} {'Facteur':>10}")
print(f"  {'-'*55}")
print(f"  {'Triplets':<25} {n_triplets_init:>10} {n_t:>10} {round(n_t/n_triplets_init, 1):>9}x")
print(f"  {'Entités':<25} {n_entities_init:>10} {n_e:>10} {round(n_e/n_entities_init, 1):>9}x")
print(f"  {'Relations':<25} {n_predicats_init:>10} {n_r:>10} {'—':>10}")

print("\n" + "=" * 60)
print("FIN DU RAPPORT DE STATISTIQUES")
print("=" * 60)

RAPPORT DE STATISTIQUES — KB Michael Jackson × Wikidata

--- 1. KB PRIVÉE INITIALE (Lab 1) ---
  Triplets extraits      : 5202
  Entités uniques        : 1136
  Prédicats uniques      : 39
  Sources crawlées       : 10
  Répartition entités    :
        MISC : 738
      PERSON : 726
         GPE : 273
         ORG : 268
  Top 5 prédicats        :
           is_related_to : 5156
                  sortir : 4
               surnommer : 4
                recevoir : 2
               participe : 2

--- 2. ALIGNEMENT DES ENTITÉS (Step 2) ---
  Entités soumises à l'alignement : 1052
  Alignées (score ≥ 0.8)          : 541 (51.4%)
    dont score exact (1.0)         : 471
    dont score approché (0.8)      : 70
  Non trouvées (score = 0.0)       : 511
  Taux d'alignement                : 51.4%

--- 3. ALIGNEMENT DES PRÉDICATS (Step 3) ---
  Triplets dans l'ontologie        : 14
  owl:equivalentProperty           : 4
  rdfs:subPropertyOf               : 4
  Prédicats privés alignés         :
    